# 03 — Collect Article Bodies

**PURPOSE**: Fetch and parse the article-detail page for every article that
has a normalized, non-excluded bracket label (the "labeled cohort"), turning
each into `article_body`, `article_body_block`, and author-metadata rows.
This is the network-calling stage of the pipeline. No embeddings, no
geocoding, no Places API — this notebook only produces block-structured
article text.

**INPUT**:
- `data/20_processed/mbn/life/article_index.parquet` (225 rows, from 02a — read-only, never overwritten)
- `data/20_processed/mbn/life/title_labels.parquet` (102 rows, from 02b)

**OUTPUT**:
- `data/00_raw/mbn/life/articles/{articleId}.html` (one raw HTML file per successfully fetched article)
- `data/20_processed/mbn/life/article_body.{parquet,csv}` (<= cohort size rows, PK=`articleId`)
- `data/20_processed/mbn/life/article_body_block.{parquet,csv}` (PK=`articleId`+`blockIndex`)
- `data/20_processed/mbn/life/article_authors.{parquet,csv}` (PK=`articleId`)
- `data/10_interim/mbn/life/fetch_attempts.{parquet,csv}` (one row per HTTP attempt, including retries — not one of the five required contracts, but required to satisfy "record every retry/timeout/status" without inventing a sixth top-level contract)

**DEPENDENCIES**: pandas, pyarrow, requests, beautifulsoup4, lxml,
`src.collection.mbn_article_collector`, `src.parsing.mbn_article_parser`,
`src.validation.checks`

**PARAMETERS**: `config/pipeline.yaml:collection.*` (timeout, retries, backoff,
rate limit) — the same rate-limit discipline used in the original crawl-phase
prototype.

**ASSUMPTIONS**: cohort = `title_labels` rows where `isExcludedEditorialLabel`
is `False` (101 of 102 rows — `반론보도` is excluded); `article_index` is the
source of truth for each article's `url`.

**SIDE EFFECTS**: makes ~101 outbound HTTP GET requests to mbn.co.kr at a
rate-limited pace (~0.4-0.7s between requests, matching the crawl-phase
convention); writes raw HTML files under `data/00_raw/**` (append-only, one
file per articleId, never deleted).

**FAIL CONDITIONS**: a fetch or parse failure for one article never raises —
it is recorded as its own `article_body` row with `parseStatus` in
{`ok`,`partial`,`failed`,`fetch_failed`} and its attempts logged in
`fetch_attempts`. The notebook only raises on primary-key violations in the
tables it assembles (a bug in this notebook, not a network condition).

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (.git marker) from {start}")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE


In [2]:
import json
from datetime import datetime, timezone

import pandas as pd
import yaml

from src.collection.mbn_article_collector import collect_article
from src.io.paths import config_path, data_dir, ensure_parent
from src.io.parquet_io import read_table, write_table
from src.validation.checks import assert_primary_key, null_counts, duplicate_report

with open(config_path(REPO_ROOT, "pipeline.yaml"), encoding="utf-8") as f:
    PIPELINE_CFG = yaml.safe_load(f)

COLLECTION_CFG = PIPELINE_CFG["collection"]
RAW_HTML_DIR = REPO_ROOT / COLLECTION_CFG["raw_html_dir"]
RAW_HTML_DIR.mkdir(parents=True, exist_ok=True)

INDEX_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "article_index")
LABELS_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "title_labels")

OUTPUT_BODY_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "article_body")
OUTPUT_BLOCK_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "article_body_block")
OUTPUT_AUTHOR_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "article_authors")
OUTPUT_ATTEMPTS_PATH = data_dir(REPO_ROOT, "10_interim", "mbn", "life", "fetch_attempts")
OUTPUT_QUALITY_PATH = data_dir(REPO_ROOT, "80_quality", "article_body_collection_report.json")

EXECUTION_TIMESTAMP = datetime.now(timezone.utc).isoformat()

article_index = read_table(INDEX_PATH)
title_labels = read_table(LABELS_PATH)
print("article_index rows:", len(article_index), "| title_labels rows:", len(title_labels))

article_index rows: 225 | title_labels rows: 102


## Build cohort: normalized label present AND not excluded

In [3]:
cohort_labels = title_labels[~title_labels["isExcludedEditorialLabel"]]
cohort = cohort_labels.merge(
    article_index[["articleId", "url", "rawTitle", "publishedAt"]], on="articleId", how="left"
)
assert cohort["url"].isna().sum() == 0, "cohort article missing url from article_index — join integrity broken"

COHORT_SIZE = len(cohort)
print("cohort size (labeled, non-excluded):", COHORT_SIZE)
cohort[["articleId", "rawBracketToken", "normalizedBracketToken", "publishedAt"]].head(5)

cohort size (labeled, non-excluded): 101


,articleId,rawBracketToken,normalizedBracketToken,publishedAt
0,5210942,AI기상캐스터,AI기상캐스터,2026-08-07 13:04
1,5210752,Season Item,Season Item,2026-08-06 17:14
2,5210310,Health Recipe,Health Recipe,2026-08-04 19:10
3,5210190,AI 기상캐스터,AI기상캐스터,2026-08-04 13:15
4,5209482,Find Dining,Find Dining,2026-07-31 14:30


## Collect: fetch + save raw HTML + parse, one article at a time, rate-limited

In [4]:
body_rows, block_rows, author_rows, attempt_rows = [], [], [], []
failures = []

for i, row in enumerate(cohort.itertuples(index=False), start=1):
    result = collect_article(
        row.articleId,
        row.url,
        raw_html_dir=RAW_HTML_DIR,
        repo_root=REPO_ROOT,
        collected_at=EXECUTION_TIMESTAMP,
    )
    attempt_rows.extend(result["attempts"])

    if not result["ok"]:
        failures.append({"articleId": row.articleId, "url": row.url, "reason": result["failureReason"]})
        body_rows.append(
            {
                "articleId": row.articleId,
                "rawHtmlPath": None,
                "rawBody": "",
                "cleanBody": "",
                "bodyLength": 0,
                "paragraphCount": 0,
                "language": "ko",
                "bodySha256": None,
                "parserVersion": None,
                "parseStatus": "fetch_failed",
                "collectedAt": EXECUTION_TIMESTAMP,
            }
        )
        author_rows.append(
            {
                "articleId": row.articleId,
                "authorRaw": None,
                "authorName": None,
                "authorEmail": None,
                "authorParseMethod": "not_found",
                "authorParseStatus": "not_found",
            }
        )
    else:
        body_rows.append(result["body_record"])
        block_rows.extend(result["block_records"])
        author_rows.append(result["author_record"])
        if result["httpStatus"] != 200 or result.get("usedFallback") or result["body_record"]["parseStatus"] != "ok":
            failures.append(
                {
                    "articleId": row.articleId,
                    "url": row.url,
                    "reason": result["failureReason"] or f"non-ok parseStatus={result['body_record']['parseStatus']}",
                }
            )

    if i % 20 == 0 or i == COHORT_SIZE:
        print(f"[{i}/{COHORT_SIZE}] collected articleId={row.articleId} ok={result['ok']}")

print("done. body_rows:", len(body_rows), "block_rows:", len(block_rows), "author_rows:", len(author_rows), "failures:", len(failures))

[20/101] collected articleId=5202864 ok=True


[40/101] collected articleId=5192642 ok=True


[60/101] collected articleId=5184957 ok=True


[80/101] collected articleId=5175290 ok=True


[100/101] collected articleId=5166117 ok=True


[101/101] collected articleId=5166010 ok=True
done. body_rows: 101 block_rows: 1619 author_rows: 101 failures: 0


## Assemble DataFrames and enforce primary-key integrity

In [5]:
article_body = pd.DataFrame.from_records(body_rows)
article_body_block = pd.DataFrame.from_records(block_rows) if block_rows else pd.DataFrame(
    columns=["articleId", "blockIndex", "blockType", "rawText", "cleanText", "htmlFragment", "sourceSelector", "parseConfidence"]
)
article_authors = pd.DataFrame.from_records(author_rows)
fetch_attempts = pd.DataFrame.from_records(attempt_rows)

assert_primary_key(article_body, ["articleId"], context="article_body")
assert_primary_key(article_body_block, ["articleId", "blockIndex"], context="article_body_block")
assert_primary_key(article_authors, ["articleId"], context="article_authors")

print("article_body:", article_body.shape)
print("article_body_block:", article_body_block.shape)
print("article_authors:", article_authors.shape)
print("fetch_attempts:", fetch_attempts.shape)

article_body: (101, 11)
article_body_block: (1619, 8)
article_authors: (101, 6)
fetch_attempts: (101, 8)


## Quality checks + write outputs (failures preserved, never dropped)

In [6]:
parse_status_counts = article_body["parseStatus"].value_counts().to_dict()
author_status_counts = article_authors["authorParseStatus"].value_counts().to_dict()
block_type_counts = article_body_block["blockType"].value_counts().to_dict() if len(article_body_block) else {}

checks = {
    "cohort_size": COHORT_SIZE,
    "row_count_article_body": int(len(article_body)),
    "row_count_matches_cohort": len(article_body) == COHORT_SIZE,
    "parse_status_counts": parse_status_counts,
    "author_status_counts": author_status_counts,
    "block_type_counts": block_type_counts,
    "failure_count": len(failures),
    "null_counts_article_body": null_counts(article_body, ["articleId", "cleanBody", "parseStatus"]),
    "duplicate_counts_article_body": duplicate_report(article_body, [["articleId"]]),
}
for k, v in checks.items():
    print(f"{k}: {v}")

blocking_issues = []
if not checks["row_count_matches_cohort"]:
    blocking_issues.append("article_body row count does not match cohort size")
quality_status = "FAIL" if blocking_issues else ("WARN" if failures else "PASS")

body_manifest = write_table(article_body, OUTPUT_BODY_PATH, required_columns=["articleId", "parseStatus"])
block_manifest = write_table(article_body_block, OUTPUT_BLOCK_PATH, required_columns=["articleId", "blockIndex", "blockType"])
author_manifest = write_table(article_authors, OUTPUT_AUTHOR_PATH, required_columns=["articleId"])
attempts_manifest = write_table(fetch_attempts, OUTPUT_ATTEMPTS_PATH, required_columns=["articleId", "attemptNumber"])

report = {
    "notebook": "03CollectArticleBodies.ipynb",
    "executionTimestamp": EXECUTION_TIMESTAMP,
    "checks": checks,
    "failures": failures,
    "blockingIssues": blocking_issues,
    "qualityStatus": quality_status,
    "outputArtifacts": {
        "article_body": body_manifest,
        "article_body_block": block_manifest,
        "article_authors": author_manifest,
        "fetch_attempts": attempts_manifest,
    },
}
ensure_parent(OUTPUT_QUALITY_PATH)
with open(OUTPUT_QUALITY_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)

print("qualityStatus:", quality_status)
print("wrote:", OUTPUT_QUALITY_PATH.relative_to(REPO_ROOT))

cohort_size: 101
row_count_article_body: 101
row_count_matches_cohort: True
parse_status_counts: {'ok': 101}
author_status_counts: {'found': 87, 'not_found': 12, 'partial': 2}
block_type_counts: {'paragraph': 1061, 'heading': 201, 'table': 168, 'image': 89, 'byline': 89, 'caption': 11}
failure_count: 0
null_counts_article_body: {'articleId': 0, 'cleanBody': 0, 'parseStatus': 0}
duplicate_counts_article_body: {'articleId': 0}
qualityStatus: PASS
wrote: data/80_quality/article_body_collection_report.json


## Closing summary

In [7]:
print("=== ROW COUNTS ===")
print({
    "article_body": len(article_body),
    "article_body_block": len(article_body_block),
    "article_authors": len(article_authors),
    "fetch_attempts": len(fetch_attempts),
})

print("=== NULL COUNTS ===")
print(checks["null_counts_article_body"])

print("=== DUPLICATES ===")
print(checks["duplicate_counts_article_body"])

print("=== QUALITY METRICS ===")
print({
    "parse_status_counts": parse_status_counts,
    "author_status_counts": author_status_counts,
    "block_type_counts": block_type_counts,
    "failure_count": len(failures),
})

print("=== OUTPUT PATH ===")
print(body_manifest["parquet_path"])
print(block_manifest["parquet_path"])
print(author_manifest["parquet_path"])
print(attempts_manifest["parquet_path"])

print("=== OUTPUT HASH ===")
print({
    "article_body": body_manifest["parquet_sha256"],
    "article_body_block": block_manifest["parquet_sha256"],
    "article_authors": author_manifest["parquet_sha256"],
    "fetch_attempts": attempts_manifest["parquet_sha256"],
})

print("=== NEXT NOTEBOOK ===")
print("03aValidateArticleBodies.ipynb")

=== ROW COUNTS ===
{'article_body': 101, 'article_body_block': 1619, 'article_authors': 101, 'fetch_attempts': 101}
=== NULL COUNTS ===
{'articleId': 0, 'cleanBody': 0, 'parseStatus': 0}
=== DUPLICATES ===
{'articleId': 0}
=== QUALITY METRICS ===
{'parse_status_counts': {'ok': 101}, 'author_status_counts': {'found': 87, 'not_found': 12, 'partial': 2}, 'block_type_counts': {'paragraph': 1061, 'heading': 201, 'table': 168, 'image': 89, 'byline': 89, 'caption': 11}, 'failure_count': 0}
=== OUTPUT PATH ===
/home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/20_processed/mbn/life/article_body.parquet
/home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/20_processed/mbn/life/article_body_block.parquet
/home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/20_processed/mbn/life/article_authors.parquet
/home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/10_interim/mbn/life/fetch_attempts.parquet
=== OUTPUT HASH ===
{'article_body': 'a44f620a5c20d3213769929833ad36cd3ed8a89898a577ec78053fdbe4f4020c'